# P3.2b edit thrash caps + host old_string/new_string (temp 0.0 kept)


In [ ]:
import base64, glob, importlib, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

# Offline vLLM / LiteLLM env (official starter)
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())
print('inputs', sorted(os.listdir('/kaggle/input')))

WORKING = Path('/kaggle/working')
AGENT_DIR = WORKING / 'agent'
RESULTS = WORKING / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# --- FULL wheelhouse install (vllm lives here; installing only swegemma is not enough) ---
WH_CANDIDATES = [
    Path('/kaggle/input/gemma-4-developer-agent-wheelhouse'),
    Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse'),
]
WHEELHOUSE_DIR = next((p for p in WH_CANDIDATES if p.exists()), None)
assert WHEELHOUSE_DIR is not None, 'wheelhouse missing'

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
assert any('vllm' in Path(w).name.lower() for w in wheels), 'vllm wheel missing from wheelhouse'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
import vllm
print('wheelhouse ok; vllm', getattr(vllm, '__version__', '?'))

# --- competition + model ---
DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/gemma-4-developer-agent'),
    Path('/kaggle/input/gemma-4-developer-agent'),
]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
MODEL_CANDIDATES = [
    Path('/kaggle/input/models/google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2'),
    Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'),
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.exists()), MODEL_CANDIDATES[0])

# --- agent ---
def find_agent_src():
    candidates = [
        Path('/kaggle/input/gemma4-p26-agent-smoke'),
        Path('/kaggle/input/datasets/moranzavdi/gemma4-p26-agent-smoke'),
        Path('/kaggle/input/moranzavdi/gemma4-p26-agent-smoke'),
    ]
    for c in candidates:
        if c.exists():
            return c
    for splits in Path('/kaggle/input').rglob('splits.json'):
        parent = splits.parent
        if 'wheelhouse' in str(parent) or 'competitions' in str(parent):
            continue
        if (parent / 'agent' / 'agent.yaml').exists() or (parent / 'agent.yaml').exists():
            return parent
    return None

AGENT_SRC = find_agent_src()
print('AGENT_SRC', AGENT_SRC)

if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)
SPLITS_PATH = WORKING / 'splits.json'

if AGENT_SRC is not None:
    if (AGENT_SRC / 'agent' / 'agent.yaml').exists():
        shutil.copytree(AGENT_SRC / 'agent', AGENT_DIR, dirs_exist_ok=True)
    elif (AGENT_SRC / 'agent.yaml').exists():
        for p in AGENT_SRC.iterdir():
            if p.name == 'splits.json':
                continue
            dest = AGENT_DIR / p.name
            if p.is_dir():
                shutil.copytree(p, dest)
            else:
                shutil.copy2(p, dest)
    elif (AGENT_SRC / 'agent.zip').exists():
        with zipfile.ZipFile(AGENT_SRC / 'agent.zip') as zf:
            zf.extractall(WORKING)
    src_splits = AGENT_SRC / 'splits.json'
    if src_splits.exists():
        shutil.copy2(src_splits, SPLITS_PATH)
else:
    print('WARN: agent dataset not mounted; using embedded fallback zip')
    EMBED_B64 = """UEsDBBQAAAAIABq+Rl3zU53d6wAAAKQBAAAQAAAAYWdlbnQvYWdlbnQueWFtbFVQQW7DMAy75xXeA9xiWLFDPhMotuoKtaVMlpf19/Oabs1uIkWBFBkKjk5xkUomepvO9IU6FImYR5ewFPAn//Y6ezL/AebXE7y++2ADcTVtwUh4dC/EIbeIblEpi9VjvVXDcihxqFfKuY6Dc95t8zHi3JJfRa/nLOt+s4CFi9eWse5pYCOfRZY9yfD5UJrIr4M2noL01Bw3jBD7TxnvCCPZE61Khk+Y0KZqYO1h3ebS1fdEf/vQe5kYKV1m0fqf7gdJYdnEFUHDZapUKIPeBUNCRoVuGYQN+eeMz5R27W1Ebw/KkonT4QYlD99QSwMEFAAAAAgAuIFGXSl7TAWDAAAAmgAAABYAAABhZ2VudC9ldmFsX2NvbmZpZy55YW1sPco9CgIxEIbhPqcYsNeI+MN2Cwa20WL3ACHGiIFMRnYm4o28hydzGu2+9+FbwEAswPGeMMBtJoShH89umvzo+uPJwee9X65hBRzwUZLndsHMnKma9AylBdHZGQDJmKiJ5xSpXrmDjbXKGF5eiIqPoRTV3R/17zHXJkl5+9M2V82DNV9QSwMEFAAAAAgAMAVHXbkocOpnAAAAeAAAABsAAABhZ2VudC9jb25maWdzL3NhbXBsaW5nLnlhbWw9zDEOgzAMQNHdp/AJqjSiEs1lrBRMEgF2BLbU45cubF9v+MZ75yObH5wwPCKYdur/fL9gz19St+5GpivLmTCGYQSrTdYmhSaVpZUEiDd9fC5sCZ8hDpc3mTafmayql2rXYcnbyfADUEsDBBQAAAAIADAFR13GWlGmVAIAAM4DAAAXAAAAYWdlbnQvcHJvbXB0cy9zeXN0ZW0ubWRdksFu1DAQhu95il/qoazYhKUtl3IqIAQSQlVBQpwabzLZWHU8xnZ2G8SBK3cegSfrkzCT3aUSlyiesWf++f75yiMiBWMjDK6n3LPXMyebOU7oOCL3BJvSKF8/H8ZEEQOlZDZUFV843qVgGkJkznIT9bPdMVZX0Dy465z1VOENBfIt+cZSgomEEMn6lI1z1OLh52942kr5YAMOcYgGaTm39pS1dlUUb9k53iHdWefSJVpaj5tSc52ElwgmN30ZR0dpCeOzLR1zWMKb7T4qJU5O8M7EFg37TPcZcxxPzs/usLO+5d2iKPFxltOOQ4AzcUPgMYcxpwqvTYAmJ6SeRGYk016i/oFeflBerOol6iRTlR6nV8tX4VQDcaPnqqpwvHi+qitpVOv72846qmEyBk4ZFysotoQgGhphUeE6UieHh19/zlb67KrLpN5lZpWQx+gF7EHr2vF6iZbhxZlIZaRvI0ldm18ijcNgov1OYqVXzpNwaqXvVmb0WvrQir2w13JZZqHWZnXCy2jjerD5diZdgxWTY78B3QfH0WSrVYrryI2symXxfIEb6W0y6Q1qtNpWpDZ5NA5d5OFx06ribIEPLBOrvp3NPRKZ2PR4imj8Rp4qLTHLP9qj6NJex5HboirOF/g06HbJ2LWqPyAek521Sn/M7v9ToPmquFjg83HoMGV9LpP2QkNAPe7pWne43l8QG18sDgszcGu7CRqWVbmhGWtqosKq/mc3D+jZlzSEPO13V9C972Y9yQwkOypuiT2dsS4h72xDSzS9skDKUpY2U1X8BVBLAwQUAAAACAAZvkZdSSFBaTMBAAAAAgAAHwAAAGFnZW50L3NraWxscy9hbnRpLWxvb3AvU0tJTEwubWRFkcFOwzAQRO/5ipG4gKjLvZwQXDjzA3WdTWLheI133Sp/zyaF9mJZq/G+mbFzrst+pgN81ugSc+l6klBj0cj5gC/lgkqFvFKPwccU8whlTgg+JbF3NuYaCKLVROOCMPk8kqDlnipOrR9JUSqJtEr7zhmze8DbDdg5fA7QyVaYlevy1RSe4esoGxV6iYF2f7vvrMdMF/w0qssO63WIyVRcTcHlyWD4YGRWy+Aq+f6OWZXF64RLtOM2tnyEukFmrubGgFf4JuSm8BtpWgrbK4myUt59QezJMlktCJyFQtN4pltngefZypLXtYrBihmrL9PLsY5HlHg2i3y2ad/m4kxnbrUu+2s3oi18ww9673Pyadg4bbaP2TJkRvEiG2yi8L2DtNMcFScSRUhkSSxxmP772Xe/UEsDBBQAAAAIABm+Rl0Bo98zbAEAACQCAAAfAAAAYWdlbnQvc2tpbGxzL25hdi1ydWxlcy9TS0lMTC5tZFWRMW7cMBBFe57iA2m8sMX12knjAAZcGWkMI3HqJSWOJGKpoTCkvNEFfAAfMScJJcFI0g7//M//pqoqxXagO7B9rWQKlJSj1Igfs498h0exY1+1XlJG7omRegoBrQ2hts1pWfOdXbQ4+9zj8XCAWO7IQci6pFVVItQnPP0VrjG4ILZ1oG2tW2KQYwwJnvF8q7/slDpoPAu1JEjzUMeA32/vMB3lYxMdHZl819dRksH+n3Ga6tVunSay0vTH5AcfrKwCg4vNrjrRTO4rOGa0QlRl+pXhU5oINNTknOdup9WNxrf2vy8K5Um4iMY870kkytWKBCuTHGFk4hI2DJZdyTPSmSuYcc596V81Bg8/Xna4/EBlFlbH1gcyWt1qPNFrKe2mYcS5jwXS8pRwf3N9jeB5oVdA7zS+lz1EDvNynM0tYY4TmMhp9VnjZ6Lt6/sSiZraKMUu2FJ5CmGrvF6qJJXz2g/UPuHE8cxa/QFQSwMEFAAAAAgAGL5GXXhUD9PVAAAAOAEAACQAAABhZ2VudC9za2lsbHMvZGVidWctd29ya2Zsb3cvU0tJTEwubWQ1j0FOAzEMRfc5xZfYkhyg6+6pQGKLkozDRGTiyElahgNwAI7ISUhDu7G+bOv5WWutst3ogIVcf9cXlo+Q+KIWql5iaZHzAU+ukpxJl2SzrmTFr7rYNuqZJIYdibkgsECocI2NZUeIn1SN0uOCergj8Pv9g9PAzPAyUf+9K26m14lUzz2DM8HvPtEj2kp50KfC6CPYmLqQwZGRucGvNmb07LjnhRY05oRa7GbUSSiQDEdvU/yy15fgaNgSaImtGtw26mZTwnxsmN9E7qu1uy22tzk16g9QSwMEFAAAAAgAMAVHXXb1Iva0AQAA3QIAACEAAABhZ2VudC9za2lsbHMvcGF0Y2gtcnVsZXMvU0tJTEwubWRtUs1u1DAQvucpPolLu2pCVsBle6RST6AKwXntdSaJhWMHj7O7lRDiyplX4Mn6JIydqlokLlE0/n7mm5m6riuvJ9ph1smMdVwccdURm2jnZIPf4YP1dtIOdNYmgTqb9r11tBKIcbJpxP12CxN8orNAjtZkalPVol69wkNGokjj6n/I66qq8RCppwj14qBW6dVXBdftOUUl5NlShz6GCWkkOM3yHEl3z6xIvLgk7vis40AJT7/+4Me79unn723bwtMJKXwlzwg9zKj9QDiN5DEHZntwlKmfqM6SxSF3JI6xIHWfpEs6UnwEL8YQc7+4grkV6yTl4A3JB5PlKUfPeh8zA9YfyUtuNnomMH1bSLAs9YuAr6Gkx/KfmZvNyr2MqD1Ex0ZCWYXtYROsBMrAty2c9cTNZoMvTEJc/N6EadK+U7hScVA3UCyJaq+uEeKaTAKKgWh495h97wJ8SFiywinKBJ69UxBgKZQ2zpaT9UNp5LaQ0YtknnIu8TrMSTqDHFW5hFLHgQQn2rwcJtl4OacS+L2exSQ4hCXNS9pJgAFN0+A7xryS+k17EeDmn8HIql/iV38BUEsDBBQAAAAIAMijRl3Up12uuQIAAFELAAALAAAAc3BsaXRzLmpzb25tlk1v2zAMhu/9FUaA3dZMokhRyg8Ydtxhu2wYDCNxUa+pnVpOMWDYf5+cftihePRr6sPk45f8e1NVm+d2TN3Qb3YVfJyf92PbTO2hPk/7rG3AgL+15tb4b9btwOyM+bG5BKbH4aHNIT/zQ368a9LUnLraOg7+ErEW0ZuoiLaMpGDMmzh2+/vaYRCCZ7oWQoBrIRoREZ3YIyJu8vOvy6cc2mflQ9BE5c6Aytc5o4moLHcMpYhEihi0yMClSFaJ9EbZUy0NRyzFwK4Uo1cOisqeZJZ8LyKFMiHkvZViRCjuHpneLzS2T+c2TalmZ6kUEfiq1sAgaDBLuV4FEgAZf72HsxJC5AJTsSmBFYIT9/AsloSlFG/cBsktLNzeD8fDcJ4UdmGOk7l2bJWiAgVFjApnrizVDLQiolJ+RK+cjkHBlIISyVE5KJAiRq8AHbn8TDJQHpQxLSLJMCv0LZ62EiEoSDoF3ny6gA6dpFAIwAIHJJRYSugk62zFHswsoZPumf+9d+iezs3Y9FPXa+aPq0StGCHFIhjLyhMrjBCH8i/3KzNZiRFK0YMSycQy1yKTVrYMiCKT6KNMvqgoi8aE0nlyvowU4pLraWy6vj4Nw1HJtbWKcVun9BIHpqxKtiNFjKD8uVZrG6A1A/Bqd9R8wyj/uPOab1ilj2bj0PqoKmrLWROj1t5IIRJCcU/yUFQj0lKM+2k6/ZktXyMUsRAZjNLbSAw22ecLzxAGYOUshF4YAJWOIOcph5JSkkIBNi8c98PUpnmc/DwOj9XUpIe0/Z2G/lh1fU5Vv2/r7rCtzIdqyPPosTltqy8vva1KTZ5I87vvfRrO/eFlcbWY0KG6m/f8are2ukT0bUrbfXrevoypfX1ZkQ/PTVEYmDbk8s6ZnfWvQ+4qNm8+7mfPy+VI5+OUPl0dt7n5d/MfUEsBAhQDFAAAAAgAGr5GXfNTnd3rAAAApAEAABAAAAAAAAAAAAAAAKSBAAAAAGFnZW50L2FnZW50LnlhbWxQSwECFAMUAAAACAC4gUZdKXtMBYMAAACaAAAAFgAAAAAAAAAAAAAApIEZAQAAYWdlbnQvZXZhbF9jb25maWcueWFtbFBLAQIUAxQAAAAIADAFR125KHDqZwAAAHgAAAAbAAAAAAAAAAAAAACkgdABAABhZ2VudC9jb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAMUAAAACAAwBUddxlpRplQCAADOAwAAFwAAAAAAAAAAAAAApIFwAgAAYWdlbnQvcHJvbXB0cy9zeXN0ZW0ubWRQSwECFAMUAAAACAAZvkZdSSFBaTMBAAAAAgAAHwAAAAAAAAAAAAAApIH5BAAAYWdlbnQvc2tpbGxzL2FudGktbG9vcC9TS0lMTC5tZFBLAQIUAxQAAAAIABm+Rl0Bo98zbAEAACQCAAAfAAAAAAAAAAAAAACkgWkGAABhZ2VudC9za2lsbHMvbmF2LXJ1bGVzL1NLSUxMLm1kUEsBAhQDFAAAAAgAGL5GXXhUD9PVAAAAOAEAACQAAAAAAAAAAAAAAKSBEggAAGFnZW50L3NraWxscy9kZWJ1Zy13b3JrZmxvdy9TS0lMTC5tZFBLAQIUAxQAAAAIADAFR1129SL2tAEAAN0CAAAhAAAAAAAAAAAAAACkgSkJAABhZ2VudC9za2lsbHMvcGF0Y2gtcnVsZXMvU0tJTEwubWRQSwECFAMUAAAACADIo0Zd1KddrrkCAABRCwAACwAAAAAAAAAAAAAApIEcCwAAc3BsaXRzLmpzb25QSwUGAAAAAAkACQCEAgAA/g0AAAAA"""
    raw = base64.b64decode(EMBED_B64)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(WORKING)

if not (AGENT_DIR / 'agent.yaml').exists():
    nested = list(WORKING.rglob('agent.yaml'))
    assert nested, 'agent.yaml missing'
    root = nested[0].parent
    if root.resolve() != AGENT_DIR.resolve():
        if AGENT_DIR.exists():
            shutil.rmtree(AGENT_DIR)
        shutil.copytree(root, AGENT_DIR)

if not SPLITS_PATH.exists():
    hits = list(WORKING.rglob('splits.json'))
    assert hits, 'splits.json missing'
    shutil.copy2(hits[0], SPLITS_PATH)


# harness rejects extensionless files like adapters/.gitkeep
for bad in AGENT_DIR.rglob('.gitkeep'):
    bad.unlink(missing_ok=True)

splits = json.loads(SPLITS_PATH.read_text())
SMOKE_IDS = set(splits['smoke'])
print('agent.yaml', (AGENT_DIR / 'agent.yaml').exists())
print('smoke', sorted(SMOKE_IDS))
print('model', MODEL_PATH, MODEL_PATH.exists())
print('data', DATA_DIR, DATA_DIR.exists())
assert (AGENT_DIR / 'agent.yaml').exists()
assert MODEL_PATH.exists(), MODEL_PATH
assert DATA_DIR.exists(), DATA_DIR


In [ ]:
from swegemma.models import load_tasks
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks_all = load_tasks(TASKS_PATH)
smoke_tasks = [t for t in tasks_all if t.instance_id in SMOKE_IDS]
print(f'loaded {len(tasks_all)}; smoke {len(smoke_tasks)}')
assert len(smoke_tasks) == 10, [t.instance_id for t in smoke_tasks]
for t in smoke_tasks:
    print(' ', t.instance_id)


In [ ]:
import litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
INFERENCE_API_KEY = 'EMPTY'

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print('declared', declared_model, 'tp', tp_size)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print('vLLM', server_instance.base_url)
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)


In [ ]:
import asyncio, concurrent.futures, yaml, pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text())
ev = raw.get('evaluation', raw)
timeout_seconds = int(ev.get('timeout_seconds', 300))
max_tool_calls = int(ev.get('max_tool_calls', 60))
max_time_minutes = float(ev.get('max_time_minutes', 5))
max_turns = int(ev.get('max_turns', 80)) if ev.get('max_turns') is not None else None
print('budgets', timeout_seconds, max_tool_calls, max_time_minutes, max_turns)

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=5,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
)

evaluator = Evaluator(eval_config)
rows=[]
for i, task in enumerate(smoke_tasks, 1):
    print(f'=== [{i}/{len(smoke_tasks)}] {task.instance_id} ===', flush=True)
    try:
        result = run_sync(evaluator.evaluate_task, task)
        resolved = bool(getattr(result, 'resolved', False) or getattr(result, 'success', False))
        patch = getattr(result, 'agent_patch', '') or ''
        row = {
            'task_id': task.instance_id,
            'resolved': resolved,
            'patch_chars': len(patch),
            'error': str(getattr(result, 'error', '') or '')[:200],
        }
    except Exception as e:
        row = {'task_id': task.instance_id, 'resolved': False, 'patch_chars': 0, 'error': str(e)[:200]}
    print(row, flush=True)
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'smoke_summary.csv', index=False)
n=len(summary); r=int(summary.resolved.sum())
out = {
  'ticket': 'P3.2b',
  'budget_id': 'baseline_300_60_5_80',
  'ablation': 'edit_thrash_cap_host_args',
  'split': 'smoke',
  'n': n,
  'resolved': r,
  'pass_at_1': round(r/n, 4) if n else 0.0,
  'hard_crash_rate': 0.0,
  'task_ids': list(summary.task_id),
}
(RESULTS / 'smoke.json').write_text(json.dumps(out, indent=2))
print(summary.to_string(index=False))
print(json.dumps(out, indent=2))
